# 04 · 최종 앙상블 → 제출 파일 (CPU로 실행 가능)

모델별 노트북(01~03, 05~07)에서 저장한 a~d 확률을 **자리별로 같은 비율(각 1/4)** 평균해 제출 파일을 만듭니다. GPU가 필요 없습니다.

**최종 조합**

```
자리 1  Qwen2.5-VL = 0.52 × 3B(01) + 0.48 × 7B(02)          # 1단계 앙상블
자리 2  Qwen3-VL-8B LoRA (03)
자리 3  Gemma 4 31B = (QLoRA(07) + 제로샷 1120토큰(05))/2    # 같은 계열은 먼저 평균
자리 4  Qwen3.5-27B = (QLoRA 추론(06) + 고해상도·선지 이동 추론(06b)) / 2
최종    = (자리1 + 자리2 + 자리3 + 자리4) / 4
```

- 가중치 탐색 없음(동일 비율). 확신도 보정·순위 투표는 리더보드에서 떨어져 사용하지 않음
- `SOURCE = "original"`: 대회 중 만든 확률 파일 사용 → 실제 제출 파일과 동일한지 확인
- `SOURCE = "repro"`: 01~03, 05~07, 06b를 다시 실행해 `repro/`에 저장된 파일 사용

In [ ]:
import os, numpy as np, pandas as pd
from google.colab import drive
drive.mount("/content/drive")
SHARE_DIR = "/content/drive/MyDrive/ssafy/ai-challenge"
E, R = f"{SHARE_DIR}/experiment_data", f"{SHARE_DIR}/repro"
P = ["prob_a","prob_b","prob_c","prob_d"]

SOURCE = "original"      # "original" 또는 "repro"
FILES = {
  "original": {
    "3b":        (f"{E}/EXP-007_train6042_resolution1280_valid_seed43.csv", f"{E}/EXP-007_train6042_resolution1280_test_probabilities_seed43.csv"),
    "7b":        (f"{E}/EXP-034_qwen7b_language_only_train1600_resolution1280_valid_seed43.csv", f"{E}/EXP-040_qwen7b_train1600_resolution1280_test_probabilities_seed43.csv"),
    "8b":        (f"{E}/EXP-102_qwen3-vl-8b-instruct_lora_r16_valid_probs.csv", f"{E}/EXP-102_qwen3-vl-8b-instruct_lora_r16_test_probs.csv"),
    "gemma1120": (f"{E}/EXP-124_gemma4-31b_t1120_valid_probs.csv", f"{E}/EXP-124_gemma4-31b_t1120_test_probs.csv"),
    "gemma_qlora": (f"{E}/EXP-125_gemma4-31b_qlora_valid_probs.csv", f"{E}/EXP-125_gemma4-31b_qlora_test_probs.csv"),
    "qwen35":    (f"{E}/EXP-123_qwen3.5-27b_qlora_valid_probs.csv", f"{E}/EXP-123_qwen3.5-27b_qlora_test_probs.csv"),
    "qwen35_px1536": (f"{E}/EXP-131_qwen3.5-27b_qlora_px1536_shift1_valid_probs.csv", f"{E}/EXP-131_qwen3.5-27b_qlora_px1536_shift1_test_probs.csv"),
  },
  "repro": {m: (f"{R}/{m}_valid_probs.csv", f"{R}/{m}_test_probs.csv") for m in ["3b","7b","8b","gemma1120","gemma_qlora","qwen35","qwen35_px1536"]},
}[SOURCE]

valid = pd.read_csv(f"{SHARE_DIR}/split/valid_split_seed43.csv")
t_ids = pd.read_csv(FILES["3b"][1]).id.values          # test id 순서 (sample_submission과 동일)
v_ids = valid.id.values
ans = valid.answer.map(dict(zip("abcd", range(4)))).values
V = {m: pd.read_csv(v).set_index("id").loc[v_ids, P].values.astype(float) for m, (v, t) in FILES.items()}
T = {m: pd.read_csv(t).set_index("id").loc[t_ids, P].values.astype(float) for m, (v, t) in FILES.items()}
for m in V: print(f"{m:12s} 단독 valid: {(V[m].argmax(1)==ans).sum()}/672")

In [ ]:
def final(D):
    slots = [0.52*D["3b"] + 0.48*D["7b"],                 # 자리 1: Qwen2.5-VL 3B·7B
             D["8b"],                                     # 자리 2: Qwen3-VL-8B
             (D["gemma_qlora"] + D["gemma1120"]) / 2,     # 자리 3: Gemma 4 31B
             (D["qwen35"] + D["qwen35_px1536"]) / 2]       # 자리 4: Qwen3.5-27B
    return sum(slots) / len(slots)

v = final(V)
print(f"최종 조합 valid: {(v.argmax(1)==ans).sum()}/672 = {(v.argmax(1)==ans).mean():.4f}   (원본 기준 654/672)")
sub = pd.DataFrame({"id": t_ids, "answer": ["abcd"[i] for i in final(T).argmax(1)]})
assert len(sub) == 6714 and sub.answer.isin(list("abcd")).all() and not sub.id.duplicated().any()
out = f"{SHARE_DIR}/submissions/FINAL_{SOURCE}.csv"
sub.to_csv(out, index=False)
print("저장:", out, "| 분포", sub.answer.value_counts().sort_index().to_dict())

ref = f"{SHARE_DIR}/submissions/EXP-135_gemmaft_g1120_q35x2.csv"   # 리더보드 0.97438 제출 파일
if os.path.exists(ref):
    print("0.97438 제출 파일과 다른 문항 수:", (pd.read_csv(ref).answer.values != sub.answer.values).sum(), "(original이면 0이어야 함)")